In [ ]:
import os
import json
import csv
import math
import random

import numpy as np
from PIL import Image, ImageOps

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
!unzip -q -o /content/drive/MyDrive/cityscapes.zip -d /content/

In [ ]:
CITYSCAPES_ROOT = "/content/cityscapes"
NATIVE_SIZE   = (1024, 2048)
TRAIN_CROP    = (768, 768)
TRACK_VAL_SIZE = (512, 1024)
FINAL_VAL_SIZE = (1024, 2048)
SCALE_RANGE   = (0.5, 2.0)     # random scale before crop (training only)

NUM_CLASSES = 19
IGNORE_INDEX = 255

BATCH_SIZE = 16
TRACK_VAL_BATCH = 4
FINAL_VAL_BATCH = 1

NUM_EPOCHS = 50
WARMUP_EPOCHS = 3

LR_BACKBONE = 5e-5
LR_HEAD = 5e-4
WD_BACKBONE = 0.05
WD_HEAD = 1e-4
LLRD = 0.65

LOVASZ_WEIGHT = 0.5
AUX_WEIGHT = 0.4
EMA_DECAY = 0.999

TTA_SCALES = (0.75, 1.0, 1.25)
TTA_HFLIP = True

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUT_DIR = "/content/dino_finetuning_outputs"
PRED_DIR = os.path.join(OUT_DIR, "predictions")
CKPT_PATH = os.path.join(OUT_DIR, "ckpt_dino_finetuning_best.pth")
HISTORY_PATH = os.path.join(OUT_DIR, "dino_finetuning_history.json")
PER_CLASS_PATH = os.path.join(OUT_DIR, "dino_finetuning_per_class.csv")
DRIVE_OUT_DIR = "/content/drive/MyDrive/dino_finetuning_outputs"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PRED_DIR, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition (102.0 GB)


In [ ]:
LABELID_TO_TRAINID = {
    7: 0, 8: 1, 11: 2, 12: 3, 13: 4, 17: 5, 19: 6, 20: 7,
    21: 8, 22: 9, 23: 10, 24: 11, 25: 12, 26: 13, 27: 14,
    28: 15, 31: 16, 32: 17, 33: 18,
}
_LABELID_LUT = np.full(256, IGNORE_INDEX, dtype=np.uint8)
for _lid, _tid in LABELID_TO_TRAINID.items():
    _LABELID_LUT[_lid] = _tid

CLASS_NAMES = [
    "road", "sidewalk", "building", "wall", "fence", "pole",
    "traffic light", "traffic sign", "vegetation", "terrain", "sky",
    "person", "rider", "car", "truck", "bus", "train",
    "motorcycle", "bicycle",
]

CLASS_COLORS = np.array([
    [128,  64, 128], [244,  35, 232], [ 70,  70,  70], [102, 102, 156],
    [190, 153, 153], [153, 153, 153], [250, 170,  30], [220, 220,   0],
    [107, 142,  35], [152, 251, 152], [ 70, 130, 180], [220,  20,  60],
    [255,   0,   0], [  0,   0, 142], [  0,   0,  70], [  0,  60, 100],
    [  0,  80, 100], [  0,   0, 230], [119,  11,  32],
], dtype=np.uint8)

def colorize_mask(mask):
    out = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls in range(NUM_CLASSES):
        out[mask == cls] = CLASS_COLORS[cls]
    return out

In [ ]:
class CityscapesDataset(Dataset):
    def __init__(self, root, split="train", val_size=NATIVE_SIZE,
                 augment=False, subset=None):
        self.root = root
        self.split = split
        self.augment = augment
        self.val_size = val_size

        img_dir = os.path.join(root, "leftImg8bit", split)
        lbl_dir = os.path.join(root, "gtFine", split)

        self.images, self.labels = [], []
        for city in sorted(os.listdir(img_dir)):
            city_dir = os.path.join(img_dir, city)
            if city.startswith(".") or not os.path.isdir(city_dir):
                continue
            for fname in sorted(os.listdir(city_dir)):
                if fname.endswith("_leftImg8bit.png"):
                    self.images.append(os.path.join(city_dir, fname))
                    self.labels.append(os.path.join(lbl_dir, city, fname.replace("_leftImg8bit.png", "_gtFine_labelIds.png")))

        if subset is not None:
            idx = np.random.choice(len(self.images), min(subset, len(self.images)), replace=False)
            self.images = [self.images[i] for i in idx]
            self.labels = [self.labels[i] for i in idx]

        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        self.color_jitter = transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1)
        print(f"  {split}: {len(self.images)} images | augment={augment} | val_size={val_size}")

    def __len__(self):
        return len(self.images)

    def _train_transform(self, img, lbl):
        H_t, W_t = TRAIN_CROP
        # random scale relative to native size
        s = random.uniform(*SCALE_RANGE)
        new_w = max(1, int(img.size[0] * s))
        new_h = max(1, int(img.size[1] * s))
        img = img.resize((new_w, new_h), Image.BILINEAR)
        lbl = lbl.resize((new_w, new_h), Image.NEAREST)

        # pad if smaller than crop (image=0, label=255 ignore)
        pad_w = max(0, W_t - new_w)
        pad_h = max(0, H_t - new_h)
        if pad_w > 0 or pad_h > 0:
            img = ImageOps.expand(img, border=(0, 0, pad_w, pad_h), fill=0)
            lbl = ImageOps.expand(lbl, border=(0, 0, pad_w, pad_h), fill=255)
            new_w += pad_w
            new_h += pad_h

        # random crop
        left = random.randint(0, new_w - W_t)
        top  = random.randint(0, new_h - H_t)
        img = img.crop((left, top, left + W_t, top + H_t))
        lbl = lbl.crop((left, top, left + W_t, top + H_t))

        # h-flip
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            lbl = lbl.transpose(Image.FLIP_LEFT_RIGHT)

        # color jitter
        img = self.color_jitter(img)
        return img, lbl

    def _val_transform(self, img, lbl):
        H, W = self.val_size
        if (img.size[1], img.size[0]) != (H, W):
            img = img.resize((W, H), Image.BILINEAR)
            lbl = lbl.resize((W, H), Image.NEAREST)
        return img, lbl

    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        lbl = Image.open(self.labels[idx])

        if self.augment:
            img, lbl = self._train_transform(img, lbl)
        else:
            img, lbl = self._val_transform(img, lbl)

        img = transforms.functional.to_tensor(img)
        img = self.normalize(img)

        lbl_arr = np.array(lbl, dtype=np.uint8)
        lbl_arr = _LABELID_LUT[lbl_arr]
        lbl = torch.from_numpy(lbl_arr).long()

        return img, lbl

In [ ]:
class DinoBackbone(nn.Module):
    def __init__(self, vit, patch_size=16, n_main=4, return_aux=True):
        super().__init__()
        self.vit = vit
        self.patch_size = patch_size
        self.n_main = n_main
        self.return_aux = return_aux
        self.embed_dim = vit.embed_dim  # 384 for ViT-S

    def forward(self, x):
        B, _, H, W = x.shape
        n = self.n_main + (1 if self.return_aux else 0)
        feats = self.vit.get_intermediate_layers(x, n=n)
        h_p = H // self.patch_size
        w_p = W // self.patch_size
        spatial = []
        for f in feats:
            f = f[:, 1:, :].reshape(B, h_p, w_p, -1).permute(0, 3, 1, 2).contiguous()
            spatial.append(f)
        if self.return_aux:
            return spatial[1:], spatial[0]
        return spatial, None


def build_dino_backbone(n_main=4, return_aux=True):
    print("  Loading DINO ViT-S/16 from torch.hub...")
    vit = torch.hub.load("facebookresearch/dino:main", "dino_vits16", pretrained=True)
    return DinoBackbone(vit, patch_size=16, n_main=n_main, return_aux=return_aux)

In [ ]:
class DPTLiteHead(nn.Module):
    def __init__(self, in_channels=384, n_layers=4, num_classes=19, proj_channels=128):
        super().__init__()
        self.proj = nn.ModuleList([nn.Conv2d(in_channels, proj_channels, kernel_size=1) for _ in range(n_layers)])
        fused = proj_channels * n_layers
        self.fusion = nn.Sequential(
            nn.Conv2d(fused, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )
        self.up1 = self._up_block(256, 128)
        self.up2 = self._up_block(128,  64)
        self.up3 = self._up_block( 64,  32)
        self.up4 = self._up_block( 32,  32)
        self.classifier = nn.Conv2d(32, num_classes, kernel_size=1)

    @staticmethod
    def _up_block(in_ch, out_ch):
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, feats):
        x = torch.cat([p(f) for p, f in zip(self.proj, feats)], dim=1)
        x = self.fusion(x)
        x = self.up1(x); x = self.up2(x); x = self.up3(x); x = self.up4(x)
        return self.classifier(x)


class AuxHead(nn.Module):
    def __init__(self, in_channels=384, num_classes=19, hidden=128, scale=16):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Conv2d(hidden, num_classes, kernel_size=1)
        self.scale = scale

    def forward(self, x):
        x = self.conv(x)
        x = self.classifier(x)
        return F.interpolate(x, scale_factor=self.scale, mode="bilinear", align_corners=False)


class SegmentationModel(nn.Module):
    def __init__(self, backbone, head, aux_head=None):
        super().__init__()
        self.backbone = backbone
        self.head = head
        self.aux_head = aux_head

    def forward(self, x):
        main_feats, aux_feat = self.backbone(x)
        main_out = self.head(main_feats)
        if self.training and self.aux_head is not None and aux_feat is not None:
            aux_out = self.aux_head(aux_feat)
            return main_out, aux_out
        return main_out


def build_model():
    backbone = build_dino_backbone(n_main=4, return_aux=True)
    head = DPTLiteHead(in_channels=backbone.embed_dim, n_layers=4, num_classes=NUM_CLASSES)
    aux  = AuxHead(in_channels=backbone.embed_dim, num_classes=NUM_CLASSES, scale=16)
    return SegmentationModel(backbone, head, aux).to(DEVICE)

In [ ]:
class IoUMeter:
    def __init__(self, num_classes, ignore_index=255):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.cm = torch.zeros(num_classes, num_classes, dtype=torch.long)

    @torch.no_grad()
    def update(self, preds, labels):
        preds = preds.flatten()
        labels = labels.flatten()
        valid = labels != self.ignore_index
        preds = preds[valid]
        labels = labels[valid]
        idx = labels * self.num_classes + preds
        binc = torch.bincount(idx, minlength=self.num_classes ** 2)
        self.cm += binc.reshape(self.num_classes, self.num_classes).cpu()

    def compute(self):
        cm = self.cm.float()
        intersection = torch.diag(cm)
        gt_total = cm.sum(dim=1)
        pred_total = cm.sum(dim=0)
        union = gt_total + pred_total - intersection
        iou = intersection / union.clamp(min=1)
        present = gt_total > 0
        miou = iou[present].mean().item() if present.any() else 0.0
        return {"miou": miou, "per_class": iou.tolist(), "present": present.tolist()}

    def reset(self):
        self.cm.zero_()


def _lovasz_grad(gt_sorted):
    p = gt_sorted.numel()
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1.0 - gt_sorted.float()).cumsum(0)
    jaccard = 1.0 - intersection / union
    if p > 1:
        jaccard[1:p] = jaccard[1:p] - jaccard[0:-1].clone()
    return jaccard


def lovasz_softmax(logits, labels, ignore_index=255):
    probs = F.softmax(logits.float(), dim=1)
    B, C, H, W = probs.shape
    probs = probs.permute(0, 2, 3, 1).reshape(-1, C)
    labels_flat = labels.reshape(-1)
    valid = labels_flat != ignore_index
    if valid.sum() == 0:
        return logits.sum() * 0.0
    probs = probs[valid]
    labels_flat = labels_flat[valid]

    losses = []
    for c in range(C):
        fg = (labels_flat == c).float()
        if fg.sum() == 0:
            continue
        class_pred = probs[:, c]
        errors = (fg - class_pred).abs()
        errors_sorted, perm = torch.sort(errors, dim=0, descending=True)
        fg_sorted = fg[perm]
        losses.append(torch.dot(errors_sorted, _lovasz_grad(fg_sorted)))
    if not losses:
        return logits.sum() * 0.0
    return torch.stack(losses).mean()


In [ ]:
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        for n, p in model.named_parameters():
            if p.dtype.is_floating_point:
                self.shadow[n] = p.detach().clone()
        for n, b in model.named_buffers():
            if b.dtype.is_floating_point:
                self.shadow[n] = b.detach().clone()

    @torch.no_grad()
    def update(self, model):
        d = self.decay
        for n, p in model.named_parameters():
            if n in self.shadow:
                self.shadow[n].mul_(d).add_(p.detach(), alpha=1.0 - d)
        for n, b in model.named_buffers():
            if n in self.shadow:
                self.shadow[n].copy_(b.detach())

    @torch.no_grad()
    def with_ema(self, model):
        backup = {}
        for n, p in model.named_parameters():
            if n in self.shadow:
                backup[n] = p.detach().clone()
                p.data.copy_(self.shadow[n])
        for n, b in model.named_buffers():
            if n in self.shadow:
                backup[n] = b.detach().clone()
                b.data.copy_(self.shadow[n])
        return backup

    @torch.no_grad()
    def restore(self, model, backup):
        for n, p in model.named_parameters():
            if n in backup:
                p.data.copy_(backup[n])
        for n, b in model.named_buffers():
            if n in backup:
                b.data.copy_(backup[n])

    def state_dict(self):
        return {k: v.detach().clone() for k, v in self.shadow.items()}

    def load_state_dict(self, state):
        for k, v in state.items():
            if k in self.shadow:
                self.shadow[k].copy_(v)

In [ ]:
def build_optimizer(model, lr_backbone, lr_head, wd_backbone, wd_head, llrd):
    no_decay_substr = ("bias", "norm", "pos_embed", "cls_token")
    def is_no_decay(name):
        return any(s in name for s in no_decay_substr)

    vit = model.backbone.vit
    n_blocks = len(vit.blocks)
    groups = []

    # head + aux_head
    head_modules = [("head", model.head)]
    if model.aux_head is not None:
        head_modules.append(("aux_head", model.aux_head))
    h_d, h_nd = [], []
    for _, mod in head_modules:
        for n, p in mod.named_parameters():
            if not p.requires_grad:
                continue
            (h_nd if is_no_decay(n) else h_d).append(p)
    if h_d: groups.append({"params": h_d, "lr": lr_head, "weight_decay": wd_head})
    if h_nd: groups.append({"params": h_nd, "lr": lr_head, "weight_decay": 0.0})

    # patch_embed + cls_token + pos_embed
    base_lr = lr_backbone * (llrd ** n_blocks)
    b_d, b_nd = [], []
    for n, p in vit.patch_embed.named_parameters():
        if not p.requires_grad:
            continue
        (b_nd if is_no_decay(n) else b_d).append(p)
    for tname in ("cls_token", "pos_embed"):
        if hasattr(vit, tname):
            t = getattr(vit, tname)
            if isinstance(t, nn.Parameter) and t.requires_grad:
                b_nd.append(t)
    if b_d:  groups.append({"params": b_d, "lr": base_lr, "weight_decay": wd_backbone})
    if b_nd: groups.append({"params": b_nd, "lr": base_lr, "weight_decay": 0.0})

    # per-block LR
    for i, block in enumerate(vit.blocks):
        block_lr = lr_backbone * (llrd ** (n_blocks - 1 - i))
        d, nd = [], []
        for n, p in block.named_parameters():
            if not p.requires_grad:
                continue
            (nd if is_no_decay(n) else d).append(p)
        if d: groups.append({"params": d, "lr": block_lr, "weight_decay": wd_backbone})
        if nd: groups.append({"params": nd, "lr": block_lr, "weight_decay": 0.0})

    if hasattr(vit, "norm"):
        norm_p = [p for p in vit.norm.parameters() if p.requires_grad]
        if norm_p:
            groups.append({"params": norm_p, "lr": lr_backbone, "weight_decay": 0.0})

    optimizer = optim.AdamW(groups)
    return optimizer, len(groups)


def build_scheduler(optimizer, num_epochs, warmup_epochs):
    warmup = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs)
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, num_epochs - warmup_epochs))
    return SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])